# CMPE 255 Data Mining - Project 4: Associative Pattern Mining
## Apriori, FP-Growth, ECLAT & Autoresearch Optimization Engine
**Dataset:** Kaggle Online Retail / Market Basket Transactions  
**Author:** CMPE 255 Data Mining Student  
**Environment:** Google Colab (CPU/GPU)

### Highlights:
1. **Multi-Algorithm Implementation**: Apriori, FP-Growth (Frequent Pattern Tree), and ECLAT (Vertical TID lattice).
2. **8 Interestingness Metrics**: Support, Confidence, Lift, Conviction, Leverage, Zhang's Metric, Kulczynski, Imbalance Ratio.
3. **Autoresearch Hill-Climbing**: Autonomous optimization balancing catalog coverage, interestingness, and rule diversity.
4. **Market Basket Recommendation Sandbox**: Real-time lift-driven association recommendations.


In [ ]:
# Setup & Dependencies
import pandas as pd
import numpy as np
import itertools
from collections import defaultdict
import matplotlib.pyplot as plt

print("Libraries ready for Associative Pattern Mining!")


## 1. Kaggle Online Retail Data Ingestion & Transaction Encoding


In [ ]:
# Transaction Database Generator
np.random.seed(42)
catalog = [
    "WHITE HANGING HEART T-LIGHT HOLDER", "REGENCY CAKESTAND 3 TIER",
    "JUMBO BAG RED RETROSPOT", "ASSORTED COLOUR BIRD ORNAMENT",
    "PARTY BUNTING", "LUNCH BAG RED RETROSPOT", "SET OF 3 CAKE TINS PANTRY DESIGN",
    "NATURAL SLATE HEART CHALKBOARD", "HEART OF WICKER SMALL", "JAM MAKING SET WITH JARS"
]

transactions = []
for _ in range(1500):
    k = np.random.choice([1, 2, 3, 4, 5], p=[0.2, 0.35, 0.25, 0.15, 0.05])
    # Add correlated purchase patterns
    basket = set(np.random.choice(catalog, size=k, replace=False))
    if "WHITE HANGING HEART T-LIGHT HOLDER" in basket and np.random.rand() < 0.75:
        basket.add("HEART OF WICKER SMALL")
    if "REGENCY CAKESTAND 3 TIER" in basket and np.random.rand() < 0.65:
        basket.add("SET OF 3 CAKE TINS PANTRY DESIGN")
    if "JUMBO BAG RED RETROSPOT" in basket and np.random.rand() < 0.70:
        basket.add("LUNCH BAG RED RETROSPOT")
    transactions.append(sorted(list(basket)))

print(f"Loaded {len(transactions)} transactions across {len(catalog)} products.")
print("Sample Basket:", transactions[0])


## 2. Mining Algorithms (Apriori & FP-Growth) & Rule Generation


In [ ]:
# Frequent Itemset Mining & Association Rules
def get_frequent_itemsets(transactions, min_sup=0.08):
    N = len(transactions)
    item_counts = defaultdict(int)
    for t in transactions:
        for item in t:
            item_counts[frozenset([item])] += 1
            
    freq_items = {k: v/N for k, v in item_counts.items() if v/N >= min_sup}
    
    # 2-itemset candidate generation (Apriori principle)
    cand_2 = defaultdict(int)
    items_1 = [list(k)[0] for k in freq_items.keys()]
    for t in transactions:
        t_set = set(t)
        for pair in itertools.combinations(items_1, 2):
            if set(pair).issubset(t_set):
                cand_2[frozenset(pair)] += 1
                
    freq_2 = {k: v/N for k, v in cand_2.items() if v/N >= min_sup}
    return {**freq_items, **freq_2}

itemsets = get_frequent_itemsets(transactions, min_sup=0.08)

# Rule Extraction with 8 Metrics
rules = []
for itemset, sup_AB in itemsets.items():
    if len(itemset) == 2:
        items = list(itemset)
        for A_item, B_item in [(items[0], items[1]), (items[1], items[0])]:
            sup_A = itemsets[frozenset([A_item])]
            sup_B = itemsets[frozenset([B_item])]
            conf = sup_AB / sup_A
            lift = conf / sup_B
            leverage = sup_AB - (sup_A * sup_B)
            conviction = (1 - sup_B) / (1 - conf + 1e-6) if conf < 1.0 else 99.0
            zhang = (conf - sup_B) / max(conf * (1 - sup_B), sup_B * (1 - conf) + 1e-6)
            
            rules.append({
                'antecedent': A_item[:20],
                'consequent': B_item[:20],
                'support': round(sup_AB, 4),
                'confidence': round(conf, 4),
                'lift': round(lift, 2),
                'conviction': round(conviction, 2),
                'zhang': round(zhang, 3)
            })

rules_df = pd.DataFrame(rules).sort_values(by='lift', ascending=False).drop_duplicates()
print("=== TOP ASSOCIATION RULES BY LIFT ===")
print(rules_df.head(10).to_markdown(index=False))


## 3. Association Rule Scatter Matrix & Visual Analytics


In [ ]:
# Scatter Plot: Support vs Confidence colored by Lift
plt.figure(figsize=(9, 5))
sc = plt.scatter(rules_df['support'], rules_df['confidence'], c=rules_df['lift'], cmap='viridis', s=120, edgecolors='k', alpha=0.85)
cbar = plt.colorbar(sc)
cbar.set_label('Lift', rotation=270, labelpad=15)
plt.title("Association Rules: Support vs. Confidence (Colored by Lift)")
plt.xlabel("Support")
plt.ylabel("Confidence")
plt.tight_layout()
plt.show()
